## Academic portfolio context

**Student / portfolio maintainer:** Mohammed Alaa Eddine GUERBOUZ  
**Institution:** INGETIS (Paris, France)  
**Programme:** Master-level studies · Generative AI / Large Language Models  
**Academic year:** 2026–2027  
**Topic:** Stateful Barista Agent with LangGraph  
**Material type:** Course practical notebook / learning exercise  

**Problem explored:** How can a conversational agent maintain state, invoke tools and progress through defined steps?

> Portfolio annotation added in October 2026. Original instructional content and attribution are preserved below. This header identifies the student maintaining this study portfolio; it is **not** a claim that all classroom template code was originally authored by the student. The source filename was `jour-3-agent-langgraph-colab-fr.ipynb`. See the folder README for requirements and limitations.


# Jour 3 - Construire un agent avec LangGraph et l'API Gemini

Bienvenue dans ce nouveau notebook du cours sur l'IA générative.

Dans ce notebook, vous allez utiliser [LangGraph](https://www.langchain.com/langgraph) pour définir une application à base de graphe et à état, construite au-dessus de l'API Gemini.

Vous allez créer un système simulé de commande dans un café, appelé BaristaBot. Il proposera aux clients une interface de chat en boucle pour commander des boissons en langage naturel. Vous construirez aussi des nœuds qui représentent le menu du café en temps réel et le système de commande « en coulisses ».

BaristaBot est utilisé dans d'autres démos de l'API Gemini. Si vous cherchez une version plus minimaliste, consultez l'[exemple BaristaBot avec appel de fonctions](https://github.com/google-gemini/cookbook/blob/main/examples/Agents_Function_Calling_Barista_Bot.ipynb), qui met en place un système similaire avec uniquement le SDK Python de l'API Gemini et l'appel de fonctions.

## **IMPORTANT !**

L'application de ce notebook récupère les **saisies de l'utilisateur** dans une **zone de texte** (la fonction [`input` de Python](https://docs.python.org/3/library/functions.html#input)). Ces appels sont commentés pour que vous puissiez utiliser **Exécution > Tout exécuter** sans interruption. Repérez bien les étapes où vous devez décommenter les appels `.invoke(...)` pour interagir avec l'application.

Si vous voulez de nouveau tout exécuter d'un coup, pensez à **recommenter** ces lignes pour que le notebook puisse tourner sans intervention.

> **Version adaptée pour Google Colab** : ce notebook est en français, lit la clé API depuis les **secrets Colab** et utilise le modèle `gemini-3.1-flash-lite` (les modèles Gemini 2.0 d'origine ont été arrêtés) avec les versions récentes de LangGraph et de LangChain. En cas de quota atteint, LangChain relance automatiquement les appels (paramètre `max_retries`). Le menu et les instructions du bot restent en anglais, mais vous pouvez commander en français : le modèle s'adapte.

## Configuration

Commencez par installer et importer le SDK LangGraph et l'intégration LangChain pour l'API Gemini.

In [ ]:
# Installe LangGraph et les paquets utilisés dans ce notebook (versions testées).
!pip install -qU "langgraph==1.2.12" "langchain-google-genai==4.4.0" "langgraph-prebuilt==1.1.0"

### Configurer votre clé API

La variable d'environnement `GOOGLE_API_KEY` permet de configurer automatiquement l'API sous-jacente. Cela fonctionne à la fois pour le SDK Python officiel de Gemini et pour LangChain/LangGraph.

Pour exécuter la cellule suivante, votre clé API doit être enregistrée dans un [secret Colab](https://colab.research.google.com/notebooks/secrets.ipynb) nommé `GOOGLE_API_KEY` :

1. Dans la barre latérale gauche de Colab, cliquez sur l'icône **clé** 🔑 (« Secrets »).
2. Cliquez sur **Ajouter un secret**, saisissez `GOOGLE_API_KEY` comme nom et collez votre clé comme valeur.
3. Activez l'interrupteur **Accès au notebook**.

Si vous n'avez pas encore de clé API, vous pouvez en obtenir une sur [AI Studio](https://aistudio.google.com/app/apikey). Vous trouverez des [instructions détaillées dans la documentation](https://ai.google.dev/gemini-api/docs/api-key).

Hors de Colab (exécution locale, VM…), la cellule utilise la variable d'environnement `GOOGLE_API_KEY`.

In [ ]:
import os

GOOGLE_API_KEY = None

# 1) Secrets Colab (icône clé dans la barre latérale gauche)
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass  # On n'est pas dans Colab

# 2) Variable d'environnement (exécution locale, VM, etc.)
if not GOOGLE_API_KEY:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError(
        "Clé API introuvable : ajoutez un secret Colab nommé GOOGLE_API_KEY "
        "et activez « Accès au notebook »."
    )

# LangChain lit la clé dans cette variable d'environnement.
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
print("Clé API chargée ✅")

Si vous obtenez une erreur du type `SecretNotFoundError` ou `NotebookAccessError`, vérifiez que le secret s'appelle exactement `GOOGLE_API_KEY` **et** que l'interrupteur **Accès au notebook** est activé dans le panneau Secrets (icône clé). Colab peut aussi afficher une fenêtre vous demandant d'autoriser l'accès : cliquez sur **Autoriser l'accès**.

## Concepts clés

Les applications LangGraph reposent sur une structure de **graphe**. En tant que développeur, vous définissez un graphe d'application qui modélise les transitions d'état de votre application. Votre application définit un schéma d'**état** (*state*), et une instance de ce schéma circule dans le graphe.

Chaque **nœud** du graphe représente une action ou une étape possible. Les nœuds modifient l'état grâce au code que vous définissez. Ces modifications peuvent venir de l'appel d'un LLM, d'une API, ou de n'importe quelle logique définie dans le nœud.

Chaque **arête** du graphe représente une transition entre états et définit le déroulement du programme. Les transitions peuvent être fixes : par exemple, dans un chatbot purement textuel dont la réponse est toujours affichée à l'utilisateur, on passe toujours de `chatbot -> user`. Elles peuvent aussi être conditionnelles, ce qui permet d'ajouter des branchements (comme un `if-else`) ou des boucles (comme `for` ou `while`).

LangGraph est très extensible et offre de nombreuses fonctionnalités qui ne font pas partie de ce tutoriel, comme la mémoire, la persistance et le streaming. Pour mieux comprendre ses concepts clés et sa philosophie, consultez la [documentation de LangGraph](https://docs.langchain.com/oss/python/langgraph/overview).

## Définir les instructions de base

L'état est un concept fondamental d'une application LangGraph. Un objet d'état est transmis à chaque nœud et à chaque transition. Ici, vous définissez un objet d'état, `OrderState`, qui contient l'historique de la conversation, une commande structurée et un indicateur qui signale si le client a fini de commander. Pour simplifier, la « structure » de la commande est une simple liste de chaînes, mais on pourrait utiliser n'importe quelle structure de données Python.

En Python, l'objet d'état de LangGraph est un [dictionnaire](https://docs.python.org/3/library/stdtypes.html#dict). Vous pouvez lui donner un schéma en le définissant comme un [`TypedDict`](https://docs.python.org/3/library/typing.html#typing.TypedDict).

Vous définissez aussi l'instruction système utilisée par le modèle Gemini. Elle fixe le ton et le style, ainsi que les règles de fonctionnement du chatbot.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class OrderState(TypedDict):
    """État représentant la conversation de commande du client."""

    # La conversation. Elle conserve l'historique entre les nœuds.
    # L'annotation `add_messages` indique à LangGraph que l'état est mis à jour
    # en ajoutant les messages renvoyés, et non en les remplaçant.
    messages: Annotated[list, add_messages]

    # La commande en cours du client.
    order: list[str]

    # Indique que la commande est passée et terminée.
    finished: bool


# L'instruction système définit le comportement attendu du chatbot : quand appeler
# les différentes fonctions, et les règles de la conversation (ton, sujets autorisés).
# Elle est laissée en anglais, comme le menu.
BARISTABOT_SYSINT = (
    "system",  # 'system' indique qu'il s'agit d'une instruction système.
    "You are a BaristaBot, an interactive cafe ordering system. A human will talk to you about the "
    "available products you have and you will answer any questions about menu items (and only about "
    "menu items - no off-topic discussion, but you can chat about the products and their history). "
    "The customer will place an order for 1 or more items from the menu, which you will structure "
    "and send to the ordering system after confirming the order with the human. "
    "\n\n"
    "Add items to the customer's order with add_to_order, and reset the order with clear_order. "
    "To see the contents of the order so far, call get_order (this is shown to you, not the user) "
    "Always confirm_order with the user (double-check) before calling place_order. Calling confirm_order will "
    "display the order items to the user and returns their response to seeing the list. Their response may contain modifications. "
    "Always verify and respond with drink and modifier names from the MENU before adding them to the order. "
    "If you are unsure a drink or modifier matches those on the MENU, ask a question to clarify or redirect. "
    "You only have the modifiers listed on the menu. "
    "Once the customer has finished ordering items, Call confirm_order to ensure it is correct then make "
    "any necessary updates and then call place_order. Once place_order has returned, thank the user and "
    "say goodbye!"
    "\n\n"
    "If any of the tools are unavailable, you can break the fourth wall and tell the user that "
    "they have not implemented them yet and should keep reading to do so.",
)

# Message d'accueil qui ouvre la conversation.
WELCOME_MSG = "Bienvenue au café BaristaBot. Tapez `q` pour quitter. Que puis-je vous servir aujourd'hui ?"

## Définir un chatbot à un seul tour

Pour illustrer le fonctionnement de LangGraph, le programme suivant définit un nœud chatbot qui exécute un seul tour de conversation avec les instructions fournies.

Chaque nœud du graphe agit sur l'objet d'état. L'état (un dictionnaire Python) est passé en paramètre au nœud (une fonction), qui renvoie le nouvel état. En pseudo-code : `state = node(state)`.

Remarque : pour le nœud `chatbot`, l'état est mis à jour en *ajoutant* le nouveau message. L'annotation `add_messages` sur `OrderState.messages` indique que les messages renvoyés par un nœud sont *ajoutés* à la suite. Normalement, l'état est mis à jour par remplacement, mais cette annotation change le comportement de `messages`.

> **Gemini 3** : le contenu d'une réponse (`msg.content`) peut être une liste de blocs (texte, signature de réflexion…). Pour afficher uniquement le texte, on utilise `msg.text`.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI

# Essayez différents modèles. Flash-Lite est rapide, économique et bon avec les outils.
# Les modèles plus gros (Flash, Pro) gèrent mieux les cas complexes à plusieurs outils,
# mais ont moins de quota gratuit.
# Comparez les fonctionnalités et les quotas ici :
#  - https://ai.google.dev/gemini-api/docs/models
MODEL = "gemini-3.1-flash-lite"

# max_retries : nombre de nouvelles tentatives en cas de quota atteint (429)
# ou de modèle surchargé (503), avec une attente croissante entre deux essais.
llm = ChatGoogleGenerativeAI(model=MODEL, max_retries=6)


def chatbot(state: OrderState) -> OrderState:
    """Le chatbot lui-même : une simple enveloppe autour de l'interface de chat du modèle."""
    message_history = [BARISTABOT_SYSINT] + state["messages"]
    return {"messages": [llm.invoke(message_history)]}


# Crée le graphe initial à partir de la définition de l'état.
graph_builder = StateGraph(OrderState)

# Ajoute la fonction chatbot au graphe, comme nœud nommé "chatbot".
graph_builder.add_node("chatbot", chatbot)

# Définit le nœud chatbot comme point d'entrée de l'application.
graph_builder.add_edge(START, "chatbot")

chat_graph = graph_builder.compile()

Il peut être utile de visualiser le graphe que vous venez de définir. Le code suivant en affiche le rendu.

In [ ]:
from IPython.display import Image, display

Image(chat_graph.get_graph().draw_mermaid_png())

Maintenant que le graphe est défini, vous pouvez l'exécuter. Il n'a qu'un nœud et une transition vers ce nœud : il passe donc de `__start__` à `chatbot`, exécute le nœud `chatbot`, puis s'arrête.

Pour exécuter le graphe, appelez `invoke` avec un objet d'état initial. Ici, il commence par le premier message de l'utilisateur.

In [ ]:
from pprint import pprint

user_msg = "Bonjour, que savez-vous faire ?"
state = chat_graph.invoke({"messages": [user_msg]})

# L'objet d'état contient beaucoup d'informations. Décommentez les lignes pprint pour tout voir.
# pprint(state)

# L'état final contient maintenant 2 messages : notre HumanMessage et un AIMessage.
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.text}")

Vous pourriez exécuter ceci dans une boucle Python, mais pour simplifier, lancez manuellement un tour de conversation supplémentaire. Ce second appel reprend l'état du premier et y ajoute un nouveau message de l'utilisateur pour obtenir une autre réponse du chatbot.

In [ ]:
user_msg = "Super, quels types de latte pouvez-vous préparer ?"

state["messages"].append(user_msg)
state = chat_graph.invoke(state)

# pprint(state)
for msg in state["messages"]:
    print(f"{type(msg).__name__}: {msg.text}")

## Ajouter un nœud humain

Au lieu d'exécuter le graphe plusieurs fois dans une boucle Python, vous pouvez utiliser LangGraph pour boucler entre les nœuds.

Le nœud `human` affiche à l'utilisateur le dernier message du LLM, puis lui demande sa saisie suivante. Ici, on utilise les fonctions Python standard `print` et `input`, mais dans un vrai café, on pourrait afficher le chat sur un écran ou le lire à voix haute, et recevoir la saisie par un micro ou un clavier tactile.

La fonction du nœud `chatbot` a aussi été mise à jour pour inclure le message d'accueil qui démarre la conversation.

In [ ]:
from langchain_core.messages.ai import AIMessage


def human_node(state: OrderState) -> OrderState:
    """Affiche le dernier message du modèle et récupère la saisie de l'utilisateur."""
    last_msg = state["messages"][-1]
    print("Modèle :", last_msg.text)

    user_input = input("Vous : ")

    # Si l'utilisateur semble vouloir quitter, on marque la conversation
    # comme terminée.
    if user_input in {"q", "quit", "exit", "goodbye", "quitter", "au revoir"}:
        state["finished"] = True

    return state | {"messages": [("user", user_input)]}


def chatbot_with_welcome_msg(state: OrderState) -> OrderState:
    """Le chatbot lui-même : une enveloppe autour de l'interface de chat du modèle."""

    if state["messages"]:
        # S'il y a des messages, on poursuit la conversation avec le modèle Gemini.
        new_output = llm.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        # Sinon, on commence par le message d'accueil.
        new_output = AIMessage(content=WELCOME_MSG)

    return state | {"messages": [new_output]}


# Construit un nouveau graphe.
graph_builder = StateGraph(OrderState)

# Ajoute les nœuds chatbot et human au graphe.
graph_builder.add_node("chatbot", chatbot_with_welcome_msg)
graph_builder.add_node("human", human_node)

# On commence toujours par le chatbot.
graph_builder.add_edge(START, "chatbot")

# Après le chatbot, on passe toujours à l'humain.
graph_builder.add_edge("chatbot", "human");

Avant d'exécuter ce graphe, remarquez que si vous ajoutiez une arête de `human` vers `chatbot`, le graphe tournerait indéfiniment, faute de condition de sortie. Une façon de casser la boucle est de vérifier si l'utilisateur a saisi `q` ou `quit`, et de sortir dans ce cas.

Dans LangGraph, on utilise pour cela une arête conditionnelle. C'est une transition comme les autres, sauf qu'une fonction personnalisée est appelée pour choisir l'arête à emprunter.

Les fonctions d'arête conditionnelle prennent l'état en entrée et renvoient une chaîne : le nom du nœud vers lequel aller.

In [ ]:
from typing import Literal


def maybe_exit_human_node(state: OrderState) -> Literal["chatbot", "__end__"]:
    """Renvoie vers le chatbot, sauf si l'utilisateur semble vouloir quitter."""
    if state.get("finished", False):
        return END
    else:
        return "chatbot"


graph_builder.add_conditional_edges("human", maybe_exit_human_node)

chat_with_human_graph = graph_builder.compile()

Image(chat_with_human_graph.get_graph().draw_mermaid_png())

Exécutez ce nouveau graphe pour voir comment la boucle d'interaction est maintenant gérée par le graphe. Tapez `q` pour quitter.

**Vous devez décommenter la ligne `.invoke(...)` pour exécuter cette étape.**

In [ ]:
# La limite de récursion par défaut pour parcourir les nœuds est de 25. L'augmenter
# permet d'essayer des commandes plus complexes, avec plusieurs étapes et allers-retours
# (et de discuter plus longtemps !)
config = {"recursion_limit": 100}

# Attention : la boucle tourne indéfiniment tant que vous ne tapez pas `q`, `quit`
# ou un autre mot de sortie défini dans `human_node`.
# Décommentez cette ligne pour exécuter le graphe :
# state = chat_with_human_graph.invoke({"messages": []}, config)

# À essayer :
#  - Discutez simplement ! Il n'y a pas encore de commande ni de menu.
#  - 'q' pour quitter.

# pprint(state)

## Ajouter un menu « en direct »

Pour l'instant, BaristaBot ne connaît pas les produits disponibles au café : il va donc inventer un menu (halluciner). Une option serait d'inscrire le menu en dur dans le prompt système. Cela fonctionnerait bien, mais pour simuler un menu plus dynamique, qui pourrait suivre l'état des stocks, vous allez placer le menu dans un outil personnalisé.

Ce système utilise deux types d'outils : des outils sans état, qui peuvent être exécutés automatiquement, et des outils avec état, qui modifient la commande. L'outil « menu actuel » est sans état : il ne modifie pas la commande en cours, il peut donc être appelé automatiquement.

Dans une application LangGraph, vous pouvez transformer des fonctions Python en outils avec le décorateur `@tool`.

In [ ]:
from langchain_core.tools import tool


@tool
def get_menu() -> str:
    """Provide the latest up-to-date menu."""
    # Ici, c'est un simple texte en dur, mais vous pourriez le connecter à une base de stocks
    # en temps réel, ou utiliser les capacités multimodales de Gemini : photographier l'ardoise
    # du café ou les produits sur le comptoir et en faire une entrée pour le modèle.

    return """
    MENU:
    Coffee Drinks:
    Espresso
    Americano
    Cold Brew

    Coffee Drinks with Milk:
    Latte
    Cappuccino
    Cortado
    Macchiato
    Mocha
    Flat White

    Tea Drinks:
    English Breakfast Tea
    Green Tea
    Earl Grey

    Tea Drinks with Milk:
    Chai Latte
    Matcha Latte
    London Fog

    Other Drinks:
    Steamer
    Hot Chocolate

    Modifiers:
    Milk options: Whole, 2%, Oat, Almond, 2% Lactose Free; Default option: whole
    Espresso shots: Single, Double, Triple, Quadruple; default: Double
    Caffeine: Decaf, Regular; default: Regular
    Hot-Iced: Hot, Iced; Default: Hot
    Sweeteners (option to add one or more): vanilla sweetener, hazelnut sweetener, caramel sauce, chocolate sauce, sugar free vanilla sweetener
    Special requests: any reasonable modification that does not involve items not on the menu, for example: 'extra hot', 'one pump', 'half caff', 'extra foam', etc.

    "dirty" means add a shot of espresso to a drink that doesn't usually have it, like "Dirty Chai Latte".
    "Regular milk" is the same as 'whole milk'.
    "Sweetened" means add some regular sugar, not a sweetener.

    Soy milk has run out of stock today, so soy is not available.
  """

Ajoutez maintenant ce nouvel outil au graphe. L'outil `get_menu` est placé dans un [`ToolNode`](https://docs.langchain.com/oss/python/langgraph/workflows-agents), qui se charge d'appeler l'outil et de transmettre sa réponse sous forme de message dans le graphe. Les outils sont aussi liés à l'objet `llm`, pour que le modèle sache qu'ils existent. Comme vous avez maintenant un autre objet `llm` à appeler, il faut mettre à jour le nœud `chatbot` pour qu'il connaisse les outils.

In [ ]:
from langgraph.prebuilt import ToolNode


# Définit les outils et crée un nœud "tools".
tools = [get_menu]
tool_node = ToolNode(tools)

# Associe les outils au modèle pour qu'il sache ce qu'il peut appeler.
llm_with_tools = llm.bind_tools(tools)


def maybe_route_to_tools(state: OrderState) -> Literal["tools", "human"]:
    """Choisit entre le nœud human et le nœud tools, selon qu'un outil est appelé ou non."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    # Le routage se base uniquement sur le dernier message.
    msg = msgs[-1]

    # Si le chatbot renvoie des tool_calls, on va vers le nœud "tools".
    if hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        return "tools"
    else:
        return "human"


def chatbot_with_tools(state: OrderState) -> OrderState:
    """Le chatbot avec outils : une simple enveloppe autour de l'interface de chat du modèle."""
    defaults = {"order": [], "finished": False}

    if state["messages"]:
        new_output = llm_with_tools.invoke([BARISTABOT_SYSINT] + state["messages"])
    else:
        new_output = AIMessage(content=WELCOME_MSG)

    # Définit des valeurs par défaut si besoin, puis transmet l'état reçu
    # en ne remplaçant que le champ "messages".
    return defaults | state | {"messages": [new_output]}


graph_builder = StateGraph(OrderState)

# Ajoute les nœuds, dont le nouveau tool_node.
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)

# Le chatbot peut aller vers tools ou vers human.
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# L'humain peut revenir au chatbot ou quitter.
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Après les outils, on revient toujours au chatbot.
graph_builder.add_edge("tools", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_menu = graph_builder.compile()

Image(graph_with_menu.get_graph().draw_mermaid_png())

Exécutez maintenant le nouveau graphe pour voir comment le modèle utilise le menu.

**Vous devez décommenter la ligne `.invoke(...)` pour exécuter cette étape.**

In [ ]:
# La commande n'est pas encore implémentée : la boucle tourne donc indéfiniment
# tant que vous ne tapez pas `q`, `quit` ou un autre mot de sortie défini dans `human_node`.
# Décommentez cette ligne pour exécuter le graphe :
# state = graph_with_menu.invoke({"messages": []}, config)

# À essayer :
# - J'aimerais une boisson à base d'espresso, qu'avez-vous ?
# - Quels thés proposez-vous ?
# - Pouvez-vous faire un long black ? (il figure au menu sous le nom « Americano » :
#   voyez si le modèle fait le lien)
# - 'q' pour quitter.


# pprint(state)

## Gérer les commandes

Pour construire une commande au fil de la conversation, vous devez mettre à jour l'état pour suivre la commande, et fournir des outils simples qui modifient cet état. Ces outils doivent être explicites : le modèle ne doit pas accéder directement à l'état interne de l'application, sinon il risquerait d'être manipulé de façon arbitraire.

Les outils de commande seront traités dans un nœud séparé, pour pouvoir modifier l'état directement. Le décorateur `@tool` reste pratique pour définir leur schéma : les outils de commande ci-dessous sont donc des fonctions Python vides.

In [ ]:
from collections.abc import Iterable
from random import randint

from langchain_core.messages.tool import ToolMessage

# Ces fonctions n'ont pas de corps : LangGraph ne permet pas aux @tool de modifier
# l'état de la conversation. Vous allez donc écrire un nœud séparé pour gérer ces mises
# à jour. @tool reste très pratique pour définir le schéma des outils : on définit donc
# des fonctions vides, liées au LLM, dont l'implémentation est déléguée à order_node.
# (Les docstrings restent en anglais : ce sont les descriptions transmises au modèle.)


@tool
def add_to_order(drink: str, modifiers: Iterable[str]) -> str:
    """Adds the specified drink to the customer's order, including any modifiers.

    Returns:
      The updated order in progress.
    """


@tool
def confirm_order() -> str:
    """Asks the customer if the order is correct.

    Returns:
      The user's free-text response.
    """


@tool
def get_order() -> str:
    """Returns the users order so far. One item per line."""


@tool
def clear_order():
    """Removes all items from the user's order."""


@tool
def place_order() -> int:
    """Sends the order to the barista for fulfillment.

    Returns:
      The estimated number of minutes until the order is ready.
    """


def order_node(state: OrderState) -> OrderState:
    """Le nœud de commande : c'est ici que l'état de la commande est modifié."""
    tool_msg = state.get("messages", [])[-1]
    order = state.get("order", [])
    outbound_msgs = []
    order_placed = False

    for tool_call in tool_msg.tool_calls:

        if tool_call["name"] == "add_to_order":

            # Chaque article est une simple chaîne, assemblée ici sous la forme "boisson (options, ...)".
            modifiers = tool_call["args"]["modifiers"]
            modifier_str = ", ".join(modifiers) if modifiers else "no modifiers"

            order.append(f'{tool_call["args"]["drink"]} ({modifier_str})')
            response = "\n".join(order)

        elif tool_call["name"] == "confirm_order":

            # On pourrait confier la confirmation au LLM, mais il vaut mieux montrer à
            # l'utilisateur les données exactes de sa commande : ce qu'il confirme correspond
            # alors exactement à ce qui part en cuisine, sans hallucination ni décalage.

            # Dans un vrai café, c'est ici qu'on brancherait l'écran de caisse pour montrer
            # la commande au client.

            print("Votre commande :")
            if not order:
                print("  (aucun article)")

            for drink in order:
                print(f"  {drink}")

            response = input("Est-ce correct ? ")

        elif tool_call["name"] == "get_order":

            response = "\n".join(order) if order else "(no order)"

        elif tool_call["name"] == "clear_order":

            order.clear()
            response = None

        elif tool_call["name"] == "place_order":

            order_text = "\n".join(order)
            print("Envoi de la commande en cuisine !")
            print(order_text)

            # TODO (vous !) : implémenter le café.
            order_placed = True
            response = randint(1, 5)  # Délai estimé en minutes

        else:
            raise NotImplementedError(f'Appel d\'outil inconnu : {tool_call["name"]}')

        # Enregistre les résultats des outils sous forme de messages d'outil.
        outbound_msgs.append(
            ToolMessage(
                content=response,
                name=tool_call["name"],
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": outbound_msgs, "order": order, "finished": order_placed}


def maybe_route_to_tools(state: OrderState) -> str:
    """Choisit entre le chat et les nœuds d'outils si un outil est appelé."""
    if not (msgs := state.get("messages", [])):
        raise ValueError(f"No messages found when parsing state: {state}")

    msg = msgs[-1]

    if state.get("finished", False):
        # Une fois la commande passée, on quitte l'application. L'instruction système
        # demande au chatbot de remercier et de dire au revoir : on peut donc sortir proprement.
        return END

    elif hasattr(msg, "tool_calls") and len(msg.tool_calls) > 0:
        # Les appels d'outils automatiques passent d'abord par le nœud `tools`.
        if any(
            tool["name"] in tool_node.tools_by_name.keys() for tool in msg.tool_calls
        ):
            return "tools"
        else:
            return "ordering"

    else:
        return "human"

Définissez maintenant le graphe. Le LLM doit aussi connaître les outils pour pouvoir les appeler. Vous créez ici 2 ensembles d'outils, correspondant aux nœuds qui les traitent : automatiques et de commande.

In [ ]:
# Les outils automatiques sont appelés automatiquement par le ToolNode
auto_tools = [get_menu]
tool_node = ToolNode(auto_tools)

# Les outils de commande sont traités par le nœud de commande.
order_tools = [add_to_order, confirm_order, get_order, clear_order, place_order]

# Le LLM doit connaître tous les outils : on les indique tous ici.
llm_with_tools = llm.bind_tools(auto_tools + order_tools)


graph_builder = StateGraph(OrderState)

# Nœuds
graph_builder.add_node("chatbot", chatbot_with_tools)
graph_builder.add_node("human", human_node)
graph_builder.add_node("tools", tool_node)
graph_builder.add_node("ordering", order_node)

# Chatbot -> {ordering, tools, human, END}
graph_builder.add_conditional_edges("chatbot", maybe_route_to_tools)
# Human -> {chatbot, END}
graph_builder.add_conditional_edges("human", maybe_exit_human_node)

# Après les outils (des deux types), on revient toujours au chatbot.
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("ordering", "chatbot")

graph_builder.add_edge(START, "chatbot")
graph_with_order_tools = graph_builder.compile()

Image(graph_with_order_tools.get_graph().draw_mermaid_png())

Exécutez maintenant le graphe complet du système de commande.

**Vous devez décommenter la ligne `.invoke(...)` pour exécuter cette étape.**

In [ ]:
# Décommentez cette ligne pour exécuter le graphe :
# state = graph_with_order_tools.invoke({"messages": []}, config)

# À essayer :
# - Commandez une boisson !
# - Modifiez votre commande.
# - « Quels thés viennent d'Angleterre ? »
# - Le graphe devrait se terminer de lui-même une fois la commande passée.

# pprint(state)

L'état de la commande a été enregistré à la fois dans la fonction `place_order` et dans l'état final de la conversation renvoyé par l'exécution du graphe. Cela montre comment intégrer vos propres systèmes dans une application à base de graphe, et comment récupérer les résultats finaux de son exécution.

In [ ]:
# Décommentez cette ligne après avoir exécuté le graphe de la cellule précédente.
# pprint(state["order"])